# 학교 계정 · Dunnhumby GCN 남은 8시드
seed49·50 완료 재사용 → seed51 최신 복구 → seed42~48. H&M 실행과 분리합니다.

실험 조건 고정: L4,300epoch,batch8192,Adam lr0.0005,uniform K1,64차원·2층,DAY1~704 학습/705~711 신규상품 test,k1.

로컬 매epoch 최신·직전 / Drive 10epoch 및 최종 저장. 런타임 소멸 시 최대9epoch 재계산. 새 전용 폴더의 검증된 오래된 저장본만 영구 정리하며 기존 파일은 유지합니다. 완료 후 최종 가중치·결과는 유지합니다. 총 여유2GB 기준, 용량조회10분 캐시. 개인용 GCN 노트북과 동시 실행 금지.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, zipfile, subprocess, torch
drive.mount('/content/drive')
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), '기존 조건과 같은 L4 GPU를 선택하세요.'
BUNDLE=Path('/content/drive/MyDrive/dh_gcn_personal_resume_inputs.zip')
assert BUNDLE.is_file(), '제공한 입력 ZIP을 학교 내 드라이브 최상위에 올려주세요.'
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''): h.update(b)
    return h.hexdigest()
assert sha(BUNDLE)=='eee4b2b5f1d969534761f0b79a032af21d6f7491e95c0fce9369514b6dcd8af2', '입력 ZIP이 제공한 원본과 다릅니다.'
ASSETS=Path('/content/gcn_personal_assets')
ASSETS.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    name='clv_dh_gcn_personal_remaining.py'
    (ASSETS/name).write_bytes(z.read(name))
sys.path.insert(0,str(ASSETS))
import clv_dh_gcn_personal_remaining as task
task.unpack_verified(BUNDLE,ASSETS)
SOURCE=task.SOURCE
REPO=Path('/content/gcn-frozen-'+SOURCE[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE
os.chdir(REPO)
sys.path.insert(0,str(REPO))
print('입력·백업 검증 완료 / GPU:',torch.cuda.get_device_name(0))
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials,_=google.auth.default()
api=build('drive','v3',credentials=credentials,cache_discovery=False)
assert api.about().get(fields='user(emailAddress)').execute(num_retries=5)['user']['emailAddress']=='jungunb@u.sogang.ac.kr'

import urllib.request
with urllib.request.urlopen('https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/976a012b7881dfadaa45bdb53c89976b3e03ea5a/clv_gcn_school_fast_storage.py',timeout=60) as r: b=r.read()
assert hashlib.sha256(b).hexdigest()=='77c4e53d566faf4e08aa7b91991315c8403358b0aa556da810cfaad5b70891aa'
(ASSETS/'clv_gcn_school_fast_storage.py').write_bytes(b)

import urllib.request
with urllib.request.urlopen('https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/976a012b7881dfadaa45bdb53c89976b3e03ea5a/clv_ngcf_school_storage.py',timeout=60) as r: b=r.read()
assert hashlib.sha256(b).hexdigest()=='0d3be22fe7d604adabdf1aaffcfbfe7d1dc748cbba4b1c7073334e0c2bd1bd4c'
(ASSETS/'clv_ngcf_school_storage.py').write_bytes(b)


## 입력 준비 · 기존 결과 보존
기존 파일을 덮어쓰지 않고 없는 복구 파일만 복사합니다. 최신 저장본은 내용의 epoch로 선택합니다.

In [ ]:
import shutil
protocol=json.loads((ASSETS/'seed51/protocol.json').read_text())
run51=Path(protocol['config']['out_dir'])/'3a93e11f5f4b'
for f in (ASSETS/'seed51').rglob('*'):
    if not f.is_file(): continue
    dest=run51/f.relative_to(ASSETS/'seed51')
    if dest.exists():
        if f.name=='protocol.json': assert json.loads(dest.read_text())==protocol
        continue
    dest.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(f,dest)
marker=run51/'personal_migration.json'
if not marker.exists(): marker.write_text(json.dumps({'source':'verified Mac recovery packet','account':'school','existing_files_preserved':True}))
ctx=task.prepare(ASSETS,'/content/drive/MyDrive/논문/data/results_v3_dunnhumby_gcn_school_final10_v1')
import clv_gcn_school_fast_storage as storage
storage.install(ctx,api)
print(json.dumps(task.report(ctx)[0],ensure_ascii=False,indent=2))


## GCN 학습 재개
로컬 매epoch, Drive 10epoch마다 저장. 완료 결과는 재학습·재평가하지 않습니다. H&M 실행은 변경하지 않습니다.

In [ ]:
status=task.run(ctx)
print(json.dumps(status,ensure_ascii=False,indent=2))

## 3. 결과 회수 (완료·중단 후 실행)
미완료는 partial ZIP으로 명시합니다. 체크포인트는 Drive에 유지하며 결과 ZIP에는 넣지 않습니다. 평균/표준편차는 80개 모두 완료 후 생성합니다. 유의성 검정은 별도입니다.

In [ ]:
from google.colab import files
output=task.export(ctx)
print('Drive 결과:',output)
files.download(output)